#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType

#Reading the bronze data

In [0]:
df = (
    spark.table("databricks_lakehouse.bronze.crm_prd_info")
)
df.display()

#Transformation of data

##Trim

In [0]:
trim_plan = {
    field.name : F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
}

df = df.withColumns(trim_plan)
df.display()

##Normalize the abbriviations 

In [0]:
df = (
    df.withColumn(
        "prd_line", F.when(F.upper(F.col("prd_line"))=="R", "Road")
        .when(F.upper(F.col("prd_line"))=="M", "Mountain")
        .when(F.upper(F.col("prd_line"))=="T", "Touring")
        .when(F.upper(F.col("prd_line"))=="S", "Other Sales")
        .otherwise("N/A")
    )
)
df.display()

##Cost cleanup 

In [0]:
# df = df.fillna(0, subset=["prd_cost"]) This is a common way to deal with null values 
df = df.withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0))) #This is proper way as it can be expanded later.
df.display()

##Product key parsing( basically I have split the prd key into cat_key and prd_key) 
these things will be entioned in the documentation in real work env.

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(F.col("prd_key"), 1, 5), "-", "_")) 
df = df.withColumn("prd_key", F.substring(F.col("prd_key"), 7, F.length(F.col("prd_key"))))  #substring stats counting from 1 rather than 0.
#this method works but it is not practical the size of each part can increase in a real world scenario.
'''
parts = F.split(F.col("prd_key"), "-")   #In pyspark split make a temp table of these parts so it contains all the values of the column. parts currently is an expressing its not doing anything F.split doesnt know whcih dataframe its using it works when we put it in a withColumn. 
df = df.withColumn("cat_id", F.concat_ws("_", parts[0], parts[1])) #concat_ws(seperator, .....)
df = df.withColumn("prd_key", F.concat_ws("-", parts[2], parts[3], parts[4]))
'''
#But in this dataset the parts are can be different so we have to use the substring method on this one.
df.display()

##Typecasting the columns if need.

In [0]:
# df.schema["prd_start_dt"].dataType
#lets say it gave string than we will convert it back to date.
df = df.withColumn("prd_start_dt", F.col("prd_start_dt").cast(DateType()))  #DateType is a pyspark datatype object
# we could also have used "date" which is an sql datatype which is undersatood by pyspark.

##Testing for null valaues

In [0]:
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).display()

# so here we are counting the number of null values in each column. for each value of c we count the nulls.
#end date null that means the product is still active.
#In case start date was null technically what should be done is that row should be put to another table. As a quarinine table.


## Renaming column names

In [0]:
rename_map = {
    field.name: field.name.replace("prd", "product").replace("dt", "date")
    for field in df.schema.fields
    if "prd" in field.name or "dt" in field.name
}
print(rename_map)

df = df.withColumnsRenamed(rename_map)
df.display()

#Write into silver table.


In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.crm_product")
)

#Checking the table

In [0]:
%sql
select * from databricks_lakehouse.silver.crm_product limit 10